#Jobberlington Json Jenerator

Just fill in the values and you can pass this into the tool directly instead of having to fill in all those search bars. Helpful for repeated runs because it's annoying to type them in over and over again.

In [ ]:
import json

## Resume

If you upload a jobberlington.json file into the tool, it will take your resume content as-is rather than summarizing it like it does when you upload a pdf on the search screen.

You can change the prompt too, if you want, and use whichever model you want to summarize.

In [ ]:
!pip install pypdf

In [ ]:
llm_model = "Qwen/Qwen2-1.5B-Instruct"

In [ ]:
import pypdf
resume_content = ""
reader = pypdf.PdfReader("resume.pdf")
number_of_pages = len(reader.pages)
for page_num in range(number_of_pages):
  page = reader.pages[page_num]
  resume_content += page.extract_text()

In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(llm_model, token=hf_token,  device_map="auto")
model = AutoModelForCausalLM.from_pretrained(llm_model,  token=hf_token,  device_map="auto")

In [ ]:
resume_prompt = """
"I will send my resume. Summarize the skills demonstrated in my experiences and output 
it in a list in bullet point format. On the first line, put a bullet for my educational status."
"""

In [ ]:
messages = [
  {"role": "system", "content": f"{resume_prompt}"},
  {"role": "user", "content": f"{resume_content}"}
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1000)
summary = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

## Search Query

In [ ]:
search_query = "Software Engineer"

## Location (Geo Id)
To find this, go to https://www.linkedin.com/jobs/ and search at the desired location. Afterwards, the url will contain a field "geoId=" where you have to get that number and put it in here.

In [ ]:
job_location = "106233382"

## Page Count
There are 25 jobs to a page. Basically equivalent to searching 25 jobs per page.

For secret quick debug testing, setting the number of pages to a decimal number also allows for even more precise job control. So 0.2 pages would be 5 jobs.

In [ ]:
page_count = 1

## Salary

0: Any Salary

1: 40k+

2: 60k+

3: 80k+

4: 100k+

5: 120k+

6: 140k+

7: 160k+

8: 180k+

9: 200k+

In [ ]:
salary = 2

## Last Posted

"Any time", "Past month", "Past week", "Past 24 hours"

In [ ]:
# job_time = "Any time"
# job_time = "Past 24 hours"
job_time = "Past week"
# job_time = "Past month"

## Experience Level

"Internship", "Entry level", "Associate", "Mid-Senior level", "Director", "Executive".

You may pick any amount of them.

In [ ]:
job_experience = ["Internship", "Entry level", "Associate", "Mid-Senior level", "Director", "Executive"]

# Custom Job Summarization Prompts

You can change the prompts used by the job summarizer and evaluator. Setting it to None just uses the default prompt.

In [ ]:
summarization_prompt = None


In [ ]:
summarization_prompt = """
I will send a job description. Summarize the required skills in the description.
"""

# Custom evaluation instructions.

You can change the prompt of the job evaluator. Setting it to None just uses the default prompt.

**Note that the default prompt currently is very specific about outputing in a json format, and the code expects an output in json format. This could break things if it doesn't work.**

In [ ]:
evaluation_prompt = None

In [ ]:
evaluation_prompt = """
    Would I be able to apply to this job? 
    Structure the output in json format, with 2 fields: 
    The first field is ANALYSIS: Do a 4 sentence analysis by seeing if my resume's skills match well to the job's skills. 
    The second field is CONFIDENCE: A single word: Answer 'HIGH' if the skills match well, 'MEDIUM' if only some skills match well, or 'LOW' if only a few skills are relevant. 
    Additionally, if the job asks for a degree greater than the one in my resume, answer 'LOW'. 
    AFTER COMPLETING THE JSON, DO NOT WRITE ANYTHING ELSE.
"""

# Generate

Just make sure you ran every cell and this will write the json for you.

In [ ]:
final_json = {
    "search_query": search_query,
    "job_location": job_location,
    "page_count": page_count,
    "salary": salary,
    "job_time": job_time,
    "job_experience": job_experience,
    "resume": summary,
    "custom_job_summarization" : summarization_prompt,
    "custom_job_evaluation" : evaluation_prompt
}
jobberlington_file = open("jobberlington.json", "w")
json.dump(final_json, jobberlington_file)
jobberlington_file.close()